In [3]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline, AutoModelForSequenceClassification
from peft import PeftModel
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from accelerate import Accelerator
import multiprocessing as mp
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_batch_response(texts, base_model, model_path, temperature, max_tokens, top_p):

    model = LLM(model=base_model,
                enable_lora=True,
                tensor_parallel_size=8,
                dtype="bfloat16",
                max_lora_rank=64)

    lora_req = LoRARequest("lora1",1,model_path)
    sampling_params = SamplingParams(max_tokens=max_tokens, temperature=temperature, top_p=top_p)

    results = model.generate(texts, sampling_params, lora_request=lora_req)

    completions = [o.outputs[0].text for o in results]

    return completions


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, top_p=0.9,
         n_samples: int = -1, input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
           
        texts = [prompt.format(json.loads(lines[i])[input_field]) for i in range(len(lines))]
        
        results = get_batch_response(
                            texts, base_model, model_path, temperature, max_tokens
                        )
        
        for result in results:
            fw.write(json.dumps(result) + '\n')


main(from_json='testsets/inspired/test.jsonl',
    to_json='test_res/inspired/llama-3.2-instruct/inspired_test_parallel.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Based on the conversation, reply 20 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n 3. [Movie Name]\n'. Then terminate the conversation. Here is the conversation: {}",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path='../outputs/sft/inspired/test',
    temperature=0.1,
    max_tokens=512,
    n_print=1,
    batch_size = 2,
    n_samples=-1)

TypeError: main() got an unexpected keyword argument 'n_print'